# 07 · Analyst Review

Measure variation across the 50 synthetic fraud analysts and compare capacity-aware assignment policies on a later alert month.

## Reading guide

This notebook is part of a connected workflow. It states the decision being made, shows the supporting checks and records limitations alongside the result. Source files are never modified in place.

In [ ]:
from pathlib import Path
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_ROOT = Path(os.environ.get("FIFAR_DATA_DIR", PROJECT_ROOT / "data" / "raw" / "FiFAR"))
REPORTS = PROJECT_ROOT / "reports"
IMAGES = PROJECT_ROOT / "images"

sns.set_theme(style="whitegrid")
CORAL = "#F08FA0"
TEAL = "#0E6268"
DARK = "#15262B"

if not DATA_ROOT.exists():
    raise FileNotFoundError(
        "Set FIFAR_DATA_DIR to the extracted official FiFAR directory before running this notebook."
    )

In [ ]:
alerts = pd.read_parquet(DATA_ROOT / "alert_data" / "processed_data" / "alerts.parquet")
predictions = pd.read_parquet(DATA_ROOT / "synthetic_experts" / "expert_predictions.parquet")
parameters = pd.read_parquet(DATA_ROOT / "synthetic_experts" / "expert_parameters.parquet")
assert alerts.index.is_unique and predictions.index.is_unique
assert set(alerts.index) == set(predictions.index)
predictions = predictions.reindex(alerts.index)
assert alerts.index.equals(predictions.index)

## 1. Analyst-level performance

In [ ]:
truth = alerts["fraud_bool"].to_numpy()
analyst_rows = []
for analyst in predictions.columns:
    decision = predictions[analyst].to_numpy()
    tp = int(((decision == 1) & (truth == 1)).sum())
    fp = int(((decision == 1) & (truth == 0)).sum())
    fn = int(((decision == 0) & (truth == 1)).sum())
    tn = int(((decision == 0) & (truth == 0)).sum())
    analyst_rows.append({
        "analyst": analyst,
        "accuracy": (tp + tn) / len(truth),
        "precision": tp / (tp + fp) if tp + fp else 0,
        "recall": tp / (tp + fn) if tp + fn else 0,
        "positive_rate": (tp + fp) / len(truth),
    })
analysts = pd.DataFrame(analyst_rows).sort_values("accuracy", ascending=False)
analysts.head(10)

In [ ]:
analysts.describe().T

Accuracy varies widely across the synthetic team. It must be interpreted with precision, recall and decision rate because the alert population remains imbalanced.

In [ ]:
fig, axis = plt.subplots(figsize=(9, 6))
sns.scatterplot(data=analysts, x="recall", y="precision", size="positive_rate", color=CORAL, alpha=.75, ax=axis)
axis.set(title="Synthetic analysts make different precision–recall trade-offs", xlabel="Recall", ylabel="Precision")
plt.show()

## 2. Capacity scenarios

In [ ]:
capacity_files = sorted((DATA_ROOT / "testbed" / "test").glob("*/capacity.csv"))
batch_files = sorted((DATA_ROOT / "testbed" / "test").glob("*/batches.csv"))
pd.Series({"test_capacity_files": len(capacity_files), "test_batch_files": len(batch_files)})

In [ ]:
pd.read_csv(capacity_files[0]).head(), pd.read_csv(batch_files[0]).head()

## 3. Historical estimation and final test

Analyst skill is estimated from alert months 3–6. Month 7 remains outside this estimation and is used to compare 25 supplied team scenarios. Score bands are fitted on the historical alerts only.

In [ ]:
review = json.loads((REPORTS / 'review_strategy_metrics.json').read_text())
pd.Series(review['data_split'])

## 4. Assignment policies

- **Random capacity:** distributes cases randomly within each active analyst's limit.
- **Global skill:** prioritises analysts with the strongest historical correctness where review is expected to improve on the screening decision.
- **Risk-band specialist:** estimates each analyst's correctness within ten historical score bands, with smoothing towards their overall result.

Every policy respects the supplied analyst capacities. Unreviewed alerts retain the screening decision.

In [ ]:
strategy = pd.DataFrame(review["strategy_summary"])
display_columns = [
    "strategy", "scenarios", "mean_accuracy", "mean_precision", "mean_recall",
    "mean_false_positive", "mean_false_negative", "mean_human_reviews",
]
strategy[display_columns].sort_values("mean_accuracy", ascending=False)

In [ ]:
plot_data = strategy.melt(
    id_vars="strategy",
    value_vars=["mean_accuracy", "mean_precision", "mean_recall"],
    var_name="metric",
    value_name="value",
)
plot_data["metric"] = plot_data["metric"].str.replace("mean_", "", regex=False).str.title()
plot_data["policy"] = plot_data["strategy"].str.replace("_", " ", regex=False).str.title()
fig, axis = plt.subplots(figsize=(11, 6))
sns.barplot(data=plot_data, x="policy", y="value", hue="metric", palette=[TEAL, CORAL, "#728489"], ax=axis)
axis.set(title="Assignment policies create different review trade-offs", xlabel="Assignment policy", ylabel="Mean across 25 scenarios")
axis.legend(title=None, frameon=False)
plt.show()

## 5. Result

Risk-band assignment achieves the highest mean accuracy and precision, while random assignment retains slightly more fraud cases. The gain over global-skill assignment is small. This is a trade-off, not evidence that a single policy is best for every operating cost.

In [ ]:
random_row = strategy.set_index("strategy").loc["random_capacity"]
specialist_row = strategy.set_index("strategy").loc["risk_band_specialist"]
pd.Series({
    "accuracy_change_vs_random_pp": (specialist_row.mean_accuracy - random_row.mean_accuracy) * 100,
    "precision_change_vs_random_pp": (specialist_row.mean_precision - random_row.mean_precision) * 100,
    "recall_change_vs_random_pp": (specialist_row.mean_recall - random_row.mean_recall) * 100,
    "mean_false_positives_avoided": random_row.mean_false_positive - specialist_row.mean_false_positive,
})

## Conclusion

Capacity and analyst selection materially change the final alert decisions. Historical specialisation can reduce unnecessary positive decisions, but the accompanying recall loss must be assessed against fraud cost and review policy.